
```
███╗   ███╗██╗███╗   ██╗███████╗
████╗ ████║██║████╗  ██║██╔════╝
██╔████╔██║██║██╔██╗ ██║█████╗
██║╚██╔╝██║██║██║╚██╗██║██╔══╝
██║ ╚═╝ ██║██║██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝╚═╝  ╚═══╝╚══════╝
```
#### developed by @Santi1303ss
---

# **1º Paso -> Crear-el-servidor (Solo hacer la primera vez)**

El codigo de abajo creará tu servidor y aceptará el EULA. Cuando uses estos scripts, Tu servidor estará listo para iniciarse.

In [ ]:
# 🔧 Instala ipywidgets si no está instalado
!pip install -q ipywidgets

# ==========================
# 📌 Sección interactiva
# ==========================
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests
import json

# Definir versiones compatibles
forge_versions = [
    "1.21.1", "1.20.1", "1.19.1", "1.18.1", "1.17.1",
    "1.16.2", "1.15.1", "1.14.2", "1.13.2", "1.12.1",
    "1.11.2", "1.10.2", "1.9.4", "1.8.8"
]

fabric_versions = [
    "1.21.1", "1.20.1", "1.19.1", "1.18.1",
    "1.17.1", "1.16.2", "1.15.1", "1.14.2"
]

# Widgets
server_type_dropdown = widgets.Dropdown(
    options=["Seleccionar", "vanilla", "forge", "fabric"],
    value="Seleccionar",
    description="Servidor:"
)

version_dropdown = widgets.Dropdown(
    options=[],
    description="Versión:"
)

confirm_button = widgets.Button(description="Confirmar selección")

output = widgets.Output()

def update_versions(change):
    if change['new'] == "forge" or change['new'] == "vanilla":
        version_dropdown.options = forge_versions
    elif change['new'] == "fabric":
        version_dropdown.options = fabric_versions
    else:
        version_dropdown.options = []

def on_confirm(b):
    clear_output()
    global server_type, version
    server_type = server_type_dropdown.value
    version = version_dropdown.value

    print(f"✅ Tipo de servidor: {server_type}")
    print(f"✅ Versión de Minecraft: {version}")

    # ==========================
    # 💾 Lógica de descarga
    # ==========================

    from google.colab import drive
    drive.mount('/content/drive')

    !mkdir -p "/content/drive/My Drive/Minecraft-server"
    %cd "/content/drive/My Drive/Minecraft-server"

    # Ajustar la versión de Forge
    forge_versions_map = {
        "1.21.1": "52.1.1",
        "1.20.1": "47.4.0",
        "1.19.1": "42.0.9",
        "1.18.1": "39.1.2",
        "1.17.1": "37.1.1",
        "1.16.2": "33.0.61",
        "1.15.1": "30.0.51",
        "1.14.2": "26.0.63",
        "1.13.2": "25.0.223",
        "1.12.1": "14.22.1.2485",
        "1.11.2": "13.20.1.2588",
        "1.10.2": "12.18.3.2511",
        "1.9.4": "12.17.0.2317",
        "1.8.8": "11.15.0.1655"
    }

    if server_type == 'forge':
        forge_version = forge_versions_map[version]
        serverURL = f"https://maven.minecraftforge.net/net/minecraftforge/forge/{version}-{forge_version}/forge-{version}-{forge_version}-installer.jar"

    elif server_type == 'vanilla':
        serverURL = f"https://mcutils.com/api/server-jars/vanilla/{version}/download"

    elif server_type == 'fabric':
        serverURL = 'https://maven.fabricmc.net/net/fabricmc/fabric-installer/1.0.1/fabric-installer-1.0.1.jar'

    jar_name = {
        'fabric': 'fabric-installer.jar',
        'forge': 'forge.jar',
        'vanilla': 'vanilla.jar'
    }

    print('\n🔻 Descargando archivo del servidor...')
    r = requests.get(serverURL)
    if r.status_code == 200:
        with open(f'/content/drive/My Drive/Minecraft-server/{jar_name[server_type]}', 'wb') as f:
            f.write(r.content)
        print('✅ Descarga completada.')
    else:
        print(f'❌ Error {r.status_code}: la versión que elegiste no está disponible.')

    if server_type == 'fabric':
        !java -jar fabric-installer.jar server -mcversion {version} -downloadMinecraft

    if server_type == 'forge':
        %cd "/content/drive/My Drive/Minecraft-server"
        !java -jar forge.jar --installServer

    # Guardar configuración
    with open("colabconfig.json", 'w') as f:
        json.dump({"server_type": server_type, "server_version": version}, f)

    # EULA
    !echo "eula=true" >> eula.txt
    print("📄 Archivo 'eula.txt' generado.")

    print("✅ Configuración finalizada.")

server_type_dropdown.observe(update_versions, names='value')
confirm_button.on_click(on_confirm)

# Mostrar la UI
display(server_type_dropdown, version_dropdown, confirm_button)


# **2º Paso -> Crear el mundo (Solo hacer la primera vez)**

Pronto te aparecera las opciones para crear tu mundo y el tipo de servidor que deseas.

In [ ]:
# ==========================
# 🌱 Configuración personalizada del mundo
# ==========================
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import json

# Leer tipo de servidor desde colabconfig
if os.path.isfile("colabconfig.json"):
    colabconfig = json.load(open("colabconfig.json"))
else:
    colabconfig = {"server_type": "vanilla"}

tipo_servidor = colabconfig.get("server_type", "vanilla")

# Widgets
nombre_mundo_widget = widgets.Text(value='world', placeholder='Nombre del mundo', description='Nombre:')
semilla_widget = widgets.Text(value='', placeholder='Deja vacío para aleatorio', description='Semilla:')
modo_widget = widgets.Dropdown(options=['survival', 'creative', 'hardcore'], value='survival', description='Modo:')
dificultad_widget = widgets.Dropdown(options=['peaceful', 'easy', 'normal', 'hard'], value='normal', description='Dificultad:')
trucos_widget = widgets.ToggleButtons(options=[('Sí', 'true'), ('No', 'false')], value='false', description='Trucos:')
bonus_widget = widgets.ToggleButtons(options=[('Sí', 'true'), ('No', 'false')], value='false', description='Bonus Chest:')
pirata_widget = widgets.ToggleButtons(options=[('Sí', 'false'), ('No', 'true')], value='false', description='¿Jugadores no oficiales?')

# Mostrar widget solo si se usa Forge, Fabric o NeoForge
crear_mods_widget = None
if tipo_servidor in ['forge', 'fabric', 'neoforge']:
    crear_mods_widget = widgets.ToggleButtons(
        options=[('Sí', True), ('No', False)],
        value=False,
        description='Crear carpeta mods:'
    )

generar_button = widgets.Button(description="Crear mundo")

# Acción del botón
def on_generate(b):
    clear_output()

    nombre_mundo = nombre_mundo_widget.value.strip() or "world"
    semilla = semilla_widget.value.strip()
    modo_juego = modo_widget.value
    dificultad = dificultad_widget.value
    trucos = trucos_widget.value
    bonus = bonus_widget.value
    online_mode = pirata_widget.value
    crear_mods = crear_mods_widget.value if crear_mods_widget else False

    gamemode_map = {"survival": "0", "creative": "1", "hardcore": "0"}
    difficulty_map = {"peaceful": "0", "easy": "1", "normal": "2", "hard": "3"}

    with open("server.properties", "w") as f:
        f.write(f"online-mode={online_mode}\n")
        f.write(f"level-name={nombre_mundo}\n")
        if semilla:
            f.write(f"level-seed={semilla}\n")
        f.write(f"gamemode={gamemode_map[modo_juego]}\n")
        f.write(f"difficulty={difficulty_map[dificultad]}\n")
        f.write("enable-command-block=true\n")
        f.write(f"allow-cheats={trucos}\n")
        f.write("generate-structures=true\n")
        f.write(f"generate-bonus-chest={bonus}\n")
        f.write(f"hardcore={'true' if modo_juego == 'hardcore' else 'false'}\n")

    if crear_mods:
        os.makedirs("mods", exist_ok=True)
        print("📁 Carpeta 'mods' creada.")

    tipo = "🟢 Premium y no oficiales" if online_mode == "false" else "🔒 Solo cuentas oficiales"
    print(f"✅ Mundo '{nombre_mundo}' configurado. Tipo de acceso: {tipo}")

# Mostrar interfaz
widgets_to_display = [
    nombre_mundo_widget, semilla_widget, modo_widget, dificultad_widget,
    trucos_widget, bonus_widget, pirata_widget
]

if crear_mods_widget:
    widgets_to_display.append(crear_mods_widget)

widgets_to_display.append(generar_button)

display(*widgets_to_display)
generar_button.on_click(on_generate)

📁 Carpeta 'mods' creada.
✅ Mundo 'Tralaleritos' configurado. Tipo de acceso: 🟢 Premium y no oficiales


# **3º Paso -> Iniciar el servidor**

Ejecutar siempre que se vaya a encernder el server


In [ ]:
# 🔧 Instalar ipywidgets si no está instalado
!pip install -q ipywidgets

# ==========================
# 🚀 Iniciar servidor
# ==========================
import os
import json
import glob
from IPython.display import clear_output
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/Minecraft-server"
!ls

# Cargar configuración
if os.path.isfile("colabconfig.json"):
  colabconfig = json.load(open("colabconfig.json"))
else:
  raise Exception("❌ No se encontró el archivo colabconfig.json. Asegúrate de haber descargado el servidor antes.")

version = colabconfig["server_version"]
server_type = colabconfig["server_type"]

# Instalar Java correcto según versión
if version < "1.17":
  !sudo apt-get purge openjdk* > /dev/null 2>&1
  !sudo apt-get install openjdk-8-jre-headless &>/dev/null && echo "Openjdk8 instalado."
elif version >= "1.20.5":
  !sudo apt-get purge openjdk* > /dev/null 2>&1
  !sudo apt-get install openjdk-21-jre-headless &>/dev/null && echo "Openjdk21 instalado."
else:
  !sudo apt-get purge openjdk* > /dev/null 2>&1
  !sudo apt-get install openjdk-17-jre-headless &>/dev/null && echo "Openjdk17 instalado."

# Verificar Java
java_ver = !java -version 2>&1 | awk -F[\"\.] -v OFS=. 'NR==1{print $2}'
print(f"Java detectado: {java_ver[0]}")

# Asignar nombre del .jar según tipo
jar_list = {
    'fabric': 'fabric-server-launch.jar',
    'forge': 'forge.jar',
    'vanilla': 'vanilla.jar'
}
jar_name = jar_list.get(server_type, "server.jar")
server_flags = ""
memory_allocation = "-Xms8192M -Xmx8192M"  # 8 GB de RAM

# ==========================
# 🚀 Iniciar con playit.gg
# ==========================
print("Instalando playit.gg...")
!curl -SsL https://playit-cloud.github.io/ppa/key.gpg | sudo apt-key add -
!sudo curl -SsL -o /etc/apt/sources.list.d/playit-cloud.list https://playit-cloud.github.io/ppa/playit-cloud.list
!sudo apt update &>/dev/null && sudo apt install playit &>/dev/null && echo "✅ Playit.gg instalado" || echo "❌ Error al instalar playit"

print("🟢 Iniciando servidor Minecraft...")

if server_type == "forge":
    if version < "1.17":
        pathlist = glob.glob(f"forge-{version}-*.jar")
        if pathlist:
            path = pathlist[0]
            !playit & java {memory_allocation} -jar "{path}" nogui
        else:
            print("❌ No se encontró forge universal.")
    else:
        pathlist = glob.glob(f"libraries/net/minecraftforge/forge/{version}-*/unix_*.txt")
        if pathlist:
            path = pathlist[0]
            !playit & java @user_jvm_args.txt "@{path}" "$@"
        else:
            print("❌ No se encontró unix_args.txt.")
else:
    !playit & java {memory_allocation} {server_flags} -jar {jar_name}